# SLAM-ASR: Türkçe Konuşma Tanıma (Turkish Speech Recognition)
### ISSAI Turkish Speech Corpus (~218 Saat) & Modern Türkçe Çok Modlu ASR

> **Referans Makale:** *An Embarrassingly Simple Approach for LLM with Strong ASR Capacity* ([arXiv:2402.08846](https://arxiv.org/abs/2402.08846))
> **Veri Seti:** ISSAI Turkish Speech Corpus (ISSAI_TSC_218) ([Hugging Face](https://huggingface.co/datasets/issai/Turkish_Speech_Corpus))

Bu notebook, Google Colab (ücretsiz T4 veya ücretli A100/L4) üzerinde **Türkçe SLAM-ASR** modelini kurmak, eğitmek ve test etmek için hazırlanmıştır.

---
### Temel Mimarisi:
* **Akustik Kodlayıcı:** `openai/whisper-small` (veya `facebook/mms-300m`) - **Dondurulmuş (Frozen)**
* **Dil Modeli (LLM):** `Qwen/Qwen2.5-7B-Instruct` (veya `Trendyol/Trendyol-LLM-7b-chat-v1.0`) - **Dondurulmuş (Frozen)**
* **Projektör:** $k=5$ Ardışık Çerçeve Birleştirme (50 Hz $\to$ 10 Hz) + Tek Lineer Katman - **Eğitilen Tek Bileşen** (~%0.6 parametre)

## 1. GPU ve Donanım Kontrolü
Colab oturumunuzun GPU hızlandırıcıya bağlı olduğunu doğrulayın (Menü: `Runtime` -> `Change runtime type` -> `T4 GPU` veya `A100 GPU`).

In [ ]:
# GPU kullanılabilirliğini kontrol edin
!nvidia-smi

import torch
print(f"[*] PyTorch Versiyonu: {torch.__version__}")
print(f"[*] CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

## 2. Depo Dosyalarını Hazırlama ve Bağımlılıkları Yükleme
GitHub reposunu çekin veya çalışma dizinine geçin ve gerekli paketleri yükleyin.

In [ ]:
import os

# Eğer doğrudan GitHub'dan klonlayacaksanız:
if not os.path.exists("tr") and not os.path.exists("slam_asr"):
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement/tr
elif os.path.exists("tr"):
    %cd tr

# Bağımlılıkları yükleyin
!pip install -q -r requirements.txt
!pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate

## 3. Kod Dosyalarının Doğrulanması ve Birim Testler
Türkçe karakter normalizasyonu (`İ -> i`, `I -> ı`), projektör katmanı, dinamik batchleme ve donanım profilleme testlerini koşturun.

In [ ]:
!python run_tests.py

## 4. Hızlı Doğrulama: Sentetik Türkçe Veri ile Test Eğitimi (30 Saniye)
21 GB'lık büyük veri setini indirmeden önce, tüm eğitim döngüsünün, gradyan akışının ve model ağırlıklarının hatasız çalıştığını sentetik verimizle test edin.

In [ ]:
# 1. Sentetik Türkçe test verisini üretin
!python scripts/create_toy_turkish_dataset.py

# 2. Hızlı doğrulama eğitimini koşturun (50 adım)
!python train.py --config config/train_toy_debug_turkish.yaml

## 5. Gerçek ISSAI_TSC_218 Türkçe Ses Korpusu Hazırlığı
Hugging Face Hub üzerinden `issai/Turkish_Speech_Corpus` veri seti çekilir (`ISSAI_TSC_218.tar.gz`).
Transkriptler Türkçe özel karakter duyarlılığıyla işlenir ve `train.jsonl`, `val.jsonl`, `test.jsonl` manifestoları oluşturulur.

> **İpucu:** İlk denemeyi daha hızlı yapmak isterseniz `--max_samples 1000` veya `--max_samples 5000` parametresi ekleyebilirsiniz.

In [ ]:
# ISSAI Turkish Speech Corpus'u indirin ve hazırlayın
!python scripts/prepare_issai_tsc.py --download --output_dir data/issai_tsc

## 6. Donanıma Göre Otomatik Optimize Olan Türkçe SLAM-ASR Eğitimi
Donanım profilleyicimiz (`device_utils`) GPU'nuzun VRAM'ine ve mimarisine göre en verimli ayarları otomatik belirler:
* **Colab T4 (16 GB) / RTX A4000 (16 GB):** 4-bit NF4 kuantizasyon + bfloat16/float16 + batch 4 + gradient accumulation 4 $\to$ Efektif Batch 16.
* **A100 (40/80 GB) / L4 (24 GB):** Yüksek throughput ve tam hassasiyet.

In [ ]:
# Donanıma göre otomatik optimize edilen Türkçe SLAM-ASR eğitimi
!python train.py --config config/hardware_adaptive_turkish.yaml

## 7. Model Değerlendirmesi (Türkçe WER ve CER Hesaplama)
Eğitilen modelin test seti üzerindeki performansını ölçün. Türkçe büyük/küçük harf ve noktalama normalizasyonu otomatik uygulanır.

In [ ]:
!python evaluate.py \
    --config config/hardware_adaptive_turkish.yaml \
    --checkpoint outputs/slam_asr_turkish_adaptive/best_checkpoint \
    --manifest data/issai_tsc/test.jsonl \
    --language tr \
    --beam_size 4

## 8. İnteraktif Çıkarım (Ses Dinleme ve Transkripsiyon)
Bir Türkçe ses kaydını dinleyin ve modelinizin konuşmayı nasıl yazıya döktüğünü canlı olarak görün.

In [ ]:
import json
import IPython.display as ipd

# Test setinden ilk örneği bulun
sample_wav = None
sample_text = ""

manifest_file = "data/issai_tsc/test.jsonl" if os.path.exists("data/issai_tsc/test.jsonl") else "data/sample_val_tr.jsonl"
if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        if first_line:
            data = json.loads(first_line)
            sample_wav = data.get("audio_filepath")
            sample_text = data.get("text", "")

if sample_wav and os.path.exists(sample_wav):
    print(f"[*] Seçilen Ses: {sample_wav}")
    print(f"[*] Gerçek Metin (Ground Truth): {sample_text}\n")
    display(ipd.Audio(sample_wav))
    
    !python inference.py \
        --audio "{sample_wav}" \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint outputs/slam_asr_turkish_adaptive/best_checkpoint/projector.pt \
        --prompt "Konuşmayı metne dönüştürün."
else:
    print("[!] Test ses dosyası bulunamadı. Lütfen ses dosyasının yolunu elle belirtin.")